In [ ]:
%pip install numpy opencv-python

# Bejeweled Blitz 自動消除｜Mac / VS Code Notebook

依序**只按各格左邊的 ▶**，不要按「Run All」。**先把 Facebook 遊戲視窗放在左邊，VS Code 在右邊；整個 8×8 棋盤必須看得見。** 校正後請勿縮放或移動瀏覽器視窗。

- **Cell 1**：匯入套件與基本設定。
- **Cell 2**：滑鼠座標校正（先左上，再右下，每次有 5 秒）。可以在玩一場測試局時校正。
- **Cell 3**：建立截圖與辨識功能，僅需執行一次。
- **Cell 4**：建立尋找最佳交換的函式，僅需執行一次。
- **Cell 5**：檢查顏色與棋盤（顯示綠點與標註圖）。
- **Cell 6**：正式自動交換，一局最多 62 秒。

**緊急停止：**滑鼠移至螢幕**最左上角**，或切回 Notebook 按 **Interrupt**（方形停止按鈕）。

若 `ModuleNotFoundError`，可先在 VS Code 終端機執行：`python3 -m pip install pyautogui numpy opencv-python pillow`。VS Code 需要 macOS「輔助使用」及「螢幕錄製」權限。

In [ ]:
import time
from pathlib import Path
import json
import pyautogui as pg
import numpy as np
import cv2
from IPython.display import display, Image as IPyImage

pg.FAILSAFE = True
pg.PAUSE = 0.01
N = 8
SYMBOL = {'red':'R','orange':'O','yellow':'Y','green':'G','blue':'B','purple':'P','white':'W','unknown':'?','empty':'.'}
HUE_CENTERS = {'red':176, 'orange':13, 'yellow':29, 'green':64, 'blue':106, 'purple':150}
CONFIG_PATH = Path.cwd() / 'bejeweled_coords.json'
ROUND_TIME = 62
DRAG_DURATION = 0.12
AFTER_MOVE_DELAY = 0.22
print('✅ Cell 1 OK：環境已準備好')

In [ ]:
def countdown_position(message, seconds=5):
    print(message, flush=True)
    for s in range(seconds, 0, -1):
        print(s, end='  ', flush=True)
        time.sleep(1)
    print()
    point = tuple(map(int, pg.position()))
    print('記錄座標：', point, flush=True)
    return point

# 每次會給 5 秒，不要點擊寶石，只要把滑鼠停在指定位置。
x0, y0 = countdown_position('❶ 將滑鼠停在棋盤「左上角第一顆寶石中心」')
x7, y7 = countdown_position('❷ 將滑鼠停在棋盤「右下角最後一顆寶石中心」')
if x7 <= x0 or y7 <= y0:
    raise ValueError('座標不正確。請重新執行這格，先左上再右下。')

cfg = {'x0': x0, 'y0': y0, 'x7': x7, 'y7': y7}
CONFIG_PATH.write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding='utf8')
print(f'✅ 校正完成：水平間距 {(x7-x0)/7:.2f} / 垂直間距 {(y7-y0)/7:.2f}')
print('記錄已存檔：', CONFIG_PATH)

In [ ]:
def gem_pos(r, c):
    x = cfg['x0'] + (cfg['x7'] - cfg['x0']) * c / 7
    y = cfg['y0'] + (cfg['y7'] - cfg['y0']) * r / 7
    return round(x), round(y)

def screenshot_bgr():
    # 使用全螢幕截圖，並按滑鼠座標尺度換算 Retina 截圖的取樣點
    rgb = np.array(pg.screenshot().convert('RGB'))
    mouse_w, mouse_h = pg.size()
    sx, sy = rgb.shape[1] / mouse_w, rgb.shape[0] / mouse_h
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR), sx, sy

def hue_distance(a, b):
    d = abs(int(a)-int(b))
    return min(d, 180-d)

def identify(patch):
    if patch.size == 0:
        return 'unknown'
    hsv = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV).reshape(-1, 3)
    bright = hsv[hsv[:,2] >= 55]
    if len(bright) == 0:
        return 'empty'
    colored = bright[(bright[:,1] >= 55) & (bright[:,2] >= 65)]
    if len(colored) / len(bright) < 0.22:
        return 'white'
    votes = {key:0 for key in HUE_CENTERS}
    for h in colored[:,0]:
        color = min(HUE_CENTERS, key=lambda k:hue_distance(h,HUE_CENTERS[k]))
        if hue_distance(h,HUE_CENTERS[color]) <= 18:
            votes[color] += 1
    winner = max(votes, key=votes.get)
    return winner if votes[winner] >= max(3, len(colored)*0.12) else 'unknown'

def read_board(debug=False):
    im, sx, sy = screenshot_bgr()
    board=[]
    for r in range(N):
        row=[]
        for c in range(N):
            mx, my = gem_pos(r,c)
            x,y = round(mx*sx),round(my*sy)
            # 取樣半徑跟 Retina 比例一致
            rad = max(5, round(8*min(sx,sy)))
            patch = im[max(0,y-rad):min(im.shape[0],y+rad+1),
                       max(0,x-rad):min(im.shape[1],x+rad+1)]
            color = identify(patch)
            row.append(color)
            if debug:
                cv2.circle(im,(x,y),max(3,round(3*sx)),(0,255,0),2)
                cv2.putText(im, SYMBOL[color], (x+7,y-6),
                            cv2.FONT_HERSHEY_SIMPLEX,0.65,(0,0,255),2)
        board.append(row)
    return (board,im,(sx,sy)) if debug else board

def show_board(board):
    print('R紅 O橘 Y黃 G綠 B藍 P紫 W白 ?未知 .空白')
    for row in board:
        print(' '.join(SYMBOL.get(x,'?') for x in row))

print('✅ Cell 3 OK：已建立畫面辨識功能')

In [ ]:
def match_cells(board):
    found=set()
    for r in range(N):
        for c in range(N):
            col=board[r][c]
            if col in ('unknown','empty'):
                continue
            if c+2 < N and all(board[r][c+i]==col for i in range(3)):
                k=c
                while k<N and board[r][k]==col:
                    found.add((r,k));k+=1
            if r+2 < N and all(board[r+i][c]==col for i in range(3)):
                k=r
                while k<N and board[k][c]==col:
                    found.add((k,c));k+=1
    return found

def best_move(board, excluded=None):
    if excluded is None:
        excluded=set()
    best=None
    for r in range(N):
        for c in range(N):
            for dr,dc in ((0,1),(1,0)):
                rr,cc=r+dr,c+dc
                if rr>=N or cc>=N:
                    continue
                a,b=(r,c),(rr,cc)
                one,two=board[r][c],board[rr][cc]
                if (a,b) in excluded or one==two or one in ('empty','unknown') or two in ('empty','unknown'):
                    continue
                test=[row.copy() for row in board]
                test[r][c],test[rr][cc] = two,one
                found=match_cells(test)
                if (a in found or b in found) and (best is None or len(found)>best[2]):
                    best=(a,b,len(found))
    return best

def swap_gems(a,b):
    x1,y1=gem_pos(*a)
    x2,y2=gem_pos(*b)
    pg.moveTo(x1,y1)
    pg.dragTo(x2,y2,duration=DRAG_DURATION,button='left')

# 不需要開啟遊戲即可測試三連消除的判斷
mock=[['red','blue','green','yellow','purple','orange','white','blue'] for _ in range(8)]
mock[0][0:3]=['red']*3
assert {(0,0),(0,1),(0,2)}.issubset(match_cells(mock))
print('✅ Cell 4 OK：消除演算法測試成功')

In [ ]:
# 在這格執行時，Facebook 棋盤一定要「完整可見」且沒有被 VS Code 蓋住。
board, img, scales = read_board(debug=True)
show_board(board)
print('截圖倍率：', tuple(round(v,2) for v in scales))
print('建議的下一步：', best_move(board))

# 儲存裁切過的標註圖。綠點要落在每顆寶石中心，紅色字母應符合顏色。
sx,sy=scales
padx=round((cfg['x7']-cfg['x0'])/14*sx)
pady=round((cfg['y7']-cfg['y0'])/14*sy)
xl=max(0,round(cfg['x0']*sx)-padx)
xr=min(img.shape[1],round(cfg['x7']*sx)+padx)
yt=max(0,round(cfg['y0']*sy)-pady)
yb=min(img.shape[0],round(cfg['y7']*sy)+pady)
if xr<=xl or yb<=yt:
    raise ValueError('棋盤裁切位置錯誤，請重新校正座標。')
marked=img[yt:yb,xl:xr]
filename=str(Path.cwd()/'board_debug.png')
cv2.imwrite(filename,marked)
print('✅ 已輸出:',filename)
display(IPyImage(filename=filename))

if sum(cell in ('unknown','empty') for row in board for cell in row)>=8:
    print('⚠️ 有不少格無法辨識，請先檢查綠點是否對在寶石中心，不要直接跑主程式。')

In [ ]:
def play_one_round():
    print('5 秒內切換至完整可見的遊戲棋盤，並開始一局。')
    print('緊急停止：滑鼠移到最左上角；或回到 VS Code 中斷執行。',flush=True)
    time.sleep(5)
    start=time.monotonic()
    count=0
    previous=None
    attempted=set()
    no_move=0
    try:
        while time.monotonic()-start < ROUND_TIME:
            board=read_board()
            bad=sum(v in ('unknown','empty') for row in board for v in row)
            if bad>=12:
                print('棋盤無法可靠辨識，停止。請重新執行 Cell 5。')
                break
            sig=tuple(map(tuple,board))
            if sig!=previous:
                attempted.clear()
                previous=sig
                no_move=0
            move=best_move(board,attempted)
            if move is None:
                no_move+=1
                if no_move>=12:
                    print('連續找不到可行交換，停止。')
                    break
                time.sleep(0.12)
                continue
            no_move=0
            a,b,score=move
            attempted.add((a,b))
            swap_gems(a,b)
            count+=1
            print(f'{time.monotonic()-start:5.1f}s 第 {count} 步：{a} → {b}，預計消 {score} 顆',flush=True)
            time.sleep(AFTER_MOVE_DELAY)
    except (KeyboardInterrupt,pg.FailSafeException):
        print('已手動停止')
    print(f'本局結束：嘗試 {count} 次交換')

play_one_round()

# 小恐龍

In [ ]:
import time
import mss
import numpy as np
import pyautogui as pyg

# ==================================
# 1. 基本設定
# ==================================

pyg.PAUSE = 0
pyg.FAILSAFE = True

# 學校 Windows 偵測範圍
START_X = 245
END_X = 640
TOP_Y = 520
BOTTOM_Y = 680

# 原本仙人掌的偵測起點
CACTUS_START_X = 290

# 保留成功的仙人掌起跳位置
CACTUS_JUMP_X = 420

# 飛鳥提前蹲下
BIRD_DUCK_X = 490

# 仙人掌與飛鳥的高度分界
CACTUS_HEIGHT_Y = 640

# 最短跳躍間隔
JUMP_DELAY = 0.45

# 記錄跳躍狀態
last_jump = 0
cactus_ready = True

# 記錄飛鳥狀態
ducking = False
duck_start = 0
last_bird_seen = 0

# 記錄白天與黑夜
last_mode = None

# ==================================
# 2. 設定截圖區域
# ==================================

monitor = {
    "left": START_X,
    "top": TOP_Y,
    "width": END_X - START_X,
    "height": BOTTOM_Y - TOP_Y
}

print("5 秒後開始，請切換到 Chrome")
time.sleep(5)

try:

    # 啟動小恐龍
    pyg.press("space")

    print("自動遊玩開始！")

    with mss.mss() as sct:

        while True:

            # ==================================
            # 3. 擷取遊戲畫面
            # ==================================

            screenshot = np.array(
                sct.grab(monitor)
            )

            gray = screenshot[:, :, 0].astype(
                np.int16
            )

            # ==================================
            # 4. 自動辨識白天與黑夜
            # ==================================

            background = int(
                np.median(
                    gray[
                        535 - TOP_Y:555 - TOP_Y,
                        -50:
                    ]
                )
            )

            if background < 128:
                mode = "黑夜"
            else:
                mode = "白天"

            if mode != last_mode:
                print("目前模式：", mode)
                last_mode = mode

            # 找出與背景不同的像素
            obstacle = (
                np.abs(gray - background) >= 60
            )

            # ==================================
            # 5. 仙人掌偵測（保留原本邏輯）
            # ==================================

            # 使用原本 X = 290 之後的範圍
            # 保留原本 Y = 535～680
            cactus_image = obstacle[
                535 - TOP_Y:,
                CACTUS_START_X - START_X:
            ]

            column_counts = np.sum(
                cactus_image, axis=0
            )

            xs = np.where(
                column_counts >= 4
            )[0]

            groups = []

            # 合併相鄰的障礙物像素
            for x in xs:

                x = int(x)

                if not groups:
                    groups.append([x])

                elif x - groups[-1][-1] <= 9:
                    groups[-1].append(x)

                else:
                    groups.append([x])

            cactus_positions = []

            for group in groups:

                left = group[0]
                right = group[-1]

                part = cactus_image[
                    :, left:right + 1
                ]

                ys = np.where(part)[0]

                if len(ys) == 0:
                    continue

                object_x = (
                    CACTUS_START_X + left
                )

                object_bottom = (
                    535 + int(np.max(ys))
                )

                # 靠近地面的物體是仙人掌
                if object_bottom >= CACTUS_HEIGHT_Y:

                    cactus_positions.append(
                        object_x
                    )

            # ==================================
            # 6. 飛鳥獨立偵測（新功能）
            # ==================================

            # 飛鳥主要活動高度
            bird_area = obstacle[
                525 - TOP_Y:635 - TOP_Y,
                :
            ]

            # 仙人掌靠近地面的部分
            lower_area = obstacle[
                640 - TOP_Y:675 - TOP_Y,
                :
            ]

            bird_columns = np.sum(
                bird_area, axis=0
            )

            lower_columns = np.sum(
                lower_area, axis=0
            )

            # 判斷哪些欄位有仙人掌
            cactus_mask = (
                lower_columns >= 6
            )

            # 排除仙人掌附近的像素
            cactus_extended = np.convolve(
                cactus_mask.astype(int),
                np.ones(61, dtype=int),
                mode="same"
            ) > 0

            # 飛鳥在上方有足夠像素
            # 但沒有與地面連接
            bird_mask = (
                (bird_columns >= 9) &
                (~cactus_extended)
            )

            # 過濾零星黑點
            bird_density = np.convolve(
                bird_mask.astype(int),
                np.ones(17, dtype=int),
                mode="same"
            )

            bird_xs = np.where(
                bird_mask &
                (bird_density >= 9)
            )[0] + START_X

            # ==================================
            # 7. 仙人掌自動跳躍
            # ==================================

            now = time.monotonic()

            cactus_near = any(
                x <= CACTUS_JUMP_X
                for x in cactus_positions
            )

            if cactus_near:

                if cactus_ready:

                    if now - last_jump >= JUMP_DELAY:

                        # 跳躍前放開蹲下鍵
                        if ducking:
                            pyg.keyUp("down")
                            ducking = False

                        # 按空白鍵跳躍
                        pyg.press("space")

                        last_jump = now
                        cactus_ready = False

                        print("仙人掌！跳躍！")

            else:

                # 仙人掌離開後準備下一跳
                cactus_ready = True

            # ==================================
            # 8. 飛鳥自動蹲下
            # ==================================

            bird_near = any(
                x <= BIRD_DUCK_X
                for x in bird_xs
            )

            # 沒有仙人掌靠近才處理飛鳥
            if bird_near and not cactus_near:

                last_bird_seen = now

                if not ducking:

                    # 持續按住向下鍵
                    pyg.keyDown("down")

                    ducking = True
                    duck_start = now

                    print("飛鳥！蹲下！")

            # ==================================
            # 9. 飛鳥通過後站起來
            # ==================================

            if ducking:

                # 至少蹲下 0.25 秒
                enough_time = (
                    now - duck_start >= 0.25
                )

                # 飛鳥消失超過 0.2 秒
                bird_gone = (
                    now - last_bird_seen >= 0.2
                )

                # 仙人掌靠近時優先站起來
                if cactus_near:

                    pyg.keyUp("down")
                    ducking = False

                # 確認飛鳥離開再站起來
                elif enough_time and bird_gone:

                    pyg.keyUp("down")
                    ducking = False

                    print("飛鳥通過！站起來！")

            # 降低 CPU 負擔
            time.sleep(0.005)

except KeyboardInterrupt:
    print("使用者停止程式")

except pyg.FailSafeException:
    print("滑鼠移到角落，緊急停止")

finally:

    # 停止時放開所有按鍵
    try:
        pyg.keyUp("down")
        pyg.keyUp("space")

    except pyg.FailSafeException:
        print("請將滑鼠移離螢幕角落")

    print("程式結束")